# 基于 YOLOv8 的 PCB 缺陷检测 — 在线演示（Colab）

使用方法：菜单栏 **Runtime → Run all** 一键运行，最后一个单元格会打印一个公开链接
（形如 `https://xxxxx.gradio.live`，约 72 小时有效），答辩当天打开即可现场演示。

> 注意：链接在笔记本保持运行期间有效；断开连接后需重新运行获取新链接。

In [ ]:
# 安装依赖（首次运行约 1-2 分钟）
!pip install ultralytics gradio -q

In [ ]:
# 下载训练好的模型权重（来自本 GitHub 仓库）
!wget -q https://github.com/jkjkjk699/PCB-Defect-Detection-YOLOv8/raw/main/weights/yolov8n_best.pt
print('权重下载完成')

In [ ]:
import cv2
import gradio as gr
from ultralytics import YOLO

model = YOLO('yolov8n_best.pt')

CLASS_NAMES_CN = {
    'Short_circuit': '短路', 'damaged': '元器件损坏', 'lack_of_part': '缺少零件',
    'miss_welding': '漏焊/虚焊', 'redundant': '多余物', 'slug': '锡渣',
    'spillover': '焊锡溢出',
}

def detect(image, conf_threshold):
    if image is None:
        return None, '请先上传一张 PCB 图像'
    result = model(image, conf=conf_threshold, verbose=False)[0]
    annotated_rgb = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    counts = {}
    for c in result.boxes.cls.tolist():
        name = result.names[int(c)]
        cn = CLASS_NAMES_CN.get(name, name)
        counts[cn] = counts.get(cn, 0) + 1
    if counts:
        summary = '检测到缺陷：' + '，'.join(f'{k} {v} 处' for k, v in counts.items())
    else:
        summary = '未检测到缺陷（该图像可能为合格品）'
    return annotated_rgb, summary

demo = gr.Interface(
    fn=detect,
    inputs=[gr.Image(type='numpy', label='上传 PCB 图像'),
            gr.Slider(0.1, 0.9, value=0.25, step=0.05, label='置信度阈值')],
    outputs=[gr.Image(type='numpy', label='检测结果'), gr.Textbox(label='检测说明')],
    title='基于 YOLOv8 的 PCB 缺陷检测',
    description='上传一张 PCB 图像，模型会自动框出缺陷位置并给出类别与数量。',
    allow_flagging='never',
)

# share=True 会生成一个公开访问链接
demo.launch(share=True)